# Dynamic Data Normalization

This notebook normalizes satellite data using ground station measurements with settings from **config.py**.

**To change settings:**
1. Edit `config.py` to change paths, dates, and pollutants
2. Run this notebook

**No need to edit this notebook!**

In [1]:
# Import necessary libraries
from pathlib import Path
import gc
import shutil
import os

from osgeo import gdal
import rioxarray
import xarray as xr
import geopandas as gpd
import pandas as pd
import numpy as np
from scipy import stats
from rasterio.enums import Resampling
import odc.geo.geom as geometry
import odc.geo.xr
import rasterio
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

In [3]:
# Load configuration
%run 00_config.ipynb
print_config()

CURRENT CONFIGURATION

Date Range:
  Start: 2024-01-01
  End: 2024-12-31

Processing Settings:
  Sample Period: 1YE
  Statistical Method: mean
  Region: All
  Pollutants: SO2, NO2, CO, O3, PM2P5, PM10

Output Path:
  C:\projects\aqi_scad\03_Output_Data\yearly_data


In [4]:
# Define paths from config
converted_zarr_path = PATHS['input']
ground_station_csv = str(PATHS['old_ground_csv'])

print(f"Input path: {converted_zarr_path}")
print(f"Ground station CSV: {ground_station_csv}")

Input path: C:\projects\aqi_scad\02_Input
Ground station CSV: C:\projects\aqi_scad\00_Ancillary_Data\EAD_Hourly_2024_AQ_Points_AQI.csv


In [5]:
def load_ground_data(ground_station_csv, pollutant):
    """
    Loads Ground station data from CSV file, cleans and prepares.
    """
    # Try different encodings
    try:
        ground_df = pd.read_csv(ground_station_csv, encoding='utf-8')
    except UnicodeDecodeError:
        try:
            ground_df = pd.read_csv(ground_station_csv, encoding='latin-1')
        except UnicodeDecodeError:
            ground_df = pd.read_csv(ground_station_csv, encoding='cp1252')
    
    pollutant_names = ['SO2', 'NO2', 'CO', 'O3', 'PM2P5', 'PM10', 'AQI']
    pollutant_names_copy = pollutant_names.copy()
    pollutant_names_copy.remove(pollutant)
    ground_df = ground_df.drop(labels=pollutant_names_copy, axis=1)
    
    # Ensure positive values
    ground_df = ground_df.loc[ground_df[pollutant] > 0]
    
    # Convert datetime
    ground_df['datetime'] = pd.to_datetime(ground_df['datetime'], format="%Y-%m-%d %H:%M:%S")
    
    # Group by month
    ground_df_grp = ground_df.groupby(['StationName', pd.Grouper(key='datetime', freq='1M')]).mean().reset_index()
    
    # Convert to GeoDataFrame with target CRS from config
    gdf = gpd.GeoDataFrame(ground_df_grp, 
                          geometry=gpd.points_from_xy(ground_df_grp['x'], ground_df_grp['y']), 
                          crs='EPSG:4326').to_crs(CONFIG['target_crs'])
    return gdf

In [6]:
def load_sat_zarr(converted_zarr_path, sat, pollutant):
    """
    Loads Satellite Pollutant Data from Zarr Files.
    """
    zarr_files = list(converted_zarr_path.glob('*.zarr'))
    
    # Create backup directory
    backup_zarr_path = converted_zarr_path / 'Backup'
    backup_zarr_path.mkdir(parents=True, exist_ok=True)
    
    da_list = []
    selected_zarr = list(filter(lambda x: f'{sat}_{pollutant}' in str(x), zarr_files))
    
    for z in selected_zarr:
        da = xr.open_zarr(z, chunks='auto')
        
        # Backup original data
        if pollutant == 'CO':
            da.to_zarr(backup_zarr_path / f"{sat}_{pollutant}_mgm3.zarr", mode='w')
        else:
            da.to_zarr(backup_zarr_path / f"{sat}_{pollutant}_ugm3.zarr", mode='w')
        
        da = da.rename({f'{list(da.keys())[0]}': f'{pollutant}'})
        da_list.append(da)
    
    # Concatenate and clean
    da_all = xr.concat(da_list, dim='time').sortby('time')
    da_all = da_all.sel(time=da_all.time.notnull())
    da_all = da_all.sel(time=~da_all.get_index("time").duplicated())
    
    # Drop CAMS-specific variables
    if sat == 'CAMS':
        da_all = da_all.drop_vars(['number', 'step', 'surface', 'valid_time'], errors='ignore')
    
    # Resample to monthly
    da_temp = da_all.resample(time='1M').mean()
    del da_all
    
    return da_temp

In [7]:
def sample_sat_data(da, gdf, pollutant):
    """
    Samples Satellite Pollutant DataArray at station locations.
    """
    x_coords = gdf.geometry.x.to_xarray()
    y_coords = gdf.geometry.y.to_xarray()
    time_coords = gdf.datetime.to_xarray()
    
    sampled = da.sel(x=x_coords, y=y_coords, time=time_coords, method='nearest')
    gdf[f"{pollutant}_Sat"] = sampled[pollutant].to_series()
    
    gdf_clean = gdf.dropna()
    return gdf_clean

In [8]:
def apply_norm(nv_df, converted_zarr_path, sat, pollutant, datetime_col):
    """
    Apply Normalization Values to DataArray using IDW interpolation.
    """
    # Load reference raster from ancillary data
    ref_raster_path = PATHS['ancillary'] / 'ad_reference_raster.tif'
    ref_raster = gdal.Open(str(ref_raster_path))
    gt = ref_raster.GetGeoTransform()
    ulx = gt[0]
    uly = gt[3]
    res = gt[1]
    
    xsize = ref_raster.RasterXSize
    ysize = ref_raster.RasterYSize
    
    lrx = ulx + xsize * res
    lry = uly - ysize * res
    
    nv_df = nv_df.reset_index()
    dates = nv_df[datetime_col].unique()
    
    # Load from BACKUP zarr files
    backup_zarr_path = converted_zarr_path / 'Backup'
    zarr_files = list(backup_zarr_path.glob('*.zarr'))
    selected_zarr = list(filter(lambda x: f'{sat}_{pollutant}' in str(x), zarr_files))
    
    da_list_orig = []
    
    for z in selected_zarr:
        ds = xr.open_zarr(z, chunks=None)
        var_name = list(ds.keys())[0]
        da = ds[var_name]
        da_list_orig.append(da)
    
    da_all = xr.concat(da_list_orig, dim='time').sortby('time')
    da_all = da_all.sel(time=da_all.time.notnull())
    da_all = da_all.sel(time=~da_all.get_index("time").duplicated())
    
    da = da_all
    da = da.rio.write_crs(CONFIG['target_crs'])
    
    da_list = []
    
    for date in dates:
        df_sub = nv_df[nv_df[datetime_col] == date]
        
        # Create IDW raster
        idw = gdal.Grid("invdist.tif", 
                       gdal.OpenEx(df_sub.to_json(), gdal.OF_VECTOR), 
                       zfield=f"{pollutant}_Norm_Val",
                       algorithm="invdist",
                       outputBounds=[ulx, uly, lrx, lry],
                       width=xsize, 
                       height=ysize)
        idw = None
        
        year = int(date.split('-')[0])
        month = int(date.split('-')[1])
        
        mask_time = ((da.time.dt.year == year) & (da.time.dt.month == month))
        da_sub = da.where(mask_time, drop=True)
        da_sub = da_sub.rio.write_crs(CONFIG['target_crs'])
        
        # Load IDW and set geospatial metadata
        xds = rioxarray.open_rasterio("invdist.tif").squeeze()
        
        from rasterio.transform import from_bounds
        transform = from_bounds(ulx, lry, lrx, uly, xsize, ysize)
        xds = xds.rio.write_transform(transform)
        xds = xds.rio.write_crs(CONFIG['target_crs'])
        
        # Reproject to match satellite data
        xds_repr_match = xds.rio.reproject_match(da_sub)
        
        # Apply normalization
        da_conv = da_sub * xds_repr_match
        da_conv = da_conv.rename(pollutant)
        da_list.append(da_conv)
    
    # Concatenate all dates
    da_all = xr.concat(da_list, dim='time').sortby('time')
    da_all = da_all.sel(time=da_all.time.notnull())
    da_all = da_all.sel(time=~da_all.get_index("time").duplicated())
    
    # Save normalized data
    if pollutant == 'CO':
        da_all.transpose('time','y', 'x').chunk('auto').to_zarr(converted_zarr_path / f"{sat}_{pollutant}_mgm3.zarr", mode='w')
    else:
        da_all.transpose('time','y', 'x').chunk('auto').to_zarr(converted_zarr_path / f"{sat}_{pollutant}_ugm3.zarr", mode='w')

## Normalize S5P Data

In [9]:
# Extract year and months from config
from datetime import datetime
start_dt = datetime.strptime(CONFIG['start_date'], '%Y-%m-%d')
end_dt = datetime.strptime(CONFIG['end_date'], '%Y-%m-%d')
year_list = list(range(start_dt.year, end_dt.year + 1))

sat = 'S5P'
s5p_pollutants = ['SO2', 'NO2', 'CO', 'O3']

print("=" * 70)
print(f"NORMALIZING {sat} DATA")
print("=" * 70)
print(f"Year range: {year_list[0]}-{year_list[-1]}")
print(f"Pollutants: {s5p_pollutants}")
print("=" * 70)

nv_df_list_m = []

for pollutant in s5p_pollutants:
    print(f"\n--- Processing {pollutant} ---")
    print("Loading ground data...")
    gdf = load_ground_data(ground_station_csv, pollutant)
    print(f"Ground stations loaded: {len(gdf)}")
    
    print("Loading satellite data...")
    da = load_sat_zarr(converted_zarr_path, sat, pollutant)
    print(f"Satellite data loaded, shape: {da[pollutant].shape}")
    
    print("Sampling satellite data at station locations...")
    gdf = sample_sat_data(da, gdf, pollutant)
    print(f"Samples matched: {len(gdf)}")
    
    print("Calculating normalization values...")
    gdf[f"{pollutant}_Norm_Val"] = (gdf[pollutant] / gdf[f"{pollutant}_Sat"])
    print(f"Norm value range: {gdf[f'{pollutant}_Norm_Val'].min():.4f} to {gdf[f'{pollutant}_Norm_Val'].max():.4f}")
    
    nv_df_list_m.append(gdf)

print("\nCombining normalization dataframes...")
nv_df_m = pd.concat(nv_df_list_m)[[f'{pollutant}_Norm_Val' for pollutant in s5p_pollutants] + ['StationName', 'datetime', 'x', 'y']]
nv_df_m['yy_mm'] = pd.to_datetime(nv_df_m['datetime']).dt.strftime('%Y-%m')
nv_df_m = nv_df_m.drop(['datetime'], axis=1)
nv_df_m = nv_df_m.groupby(['yy_mm', 'StationName']).mean()

print(f"Normalization values calculated for {len(nv_df_m)} station-month combinations")

csv_path = converted_zarr_path / f'Ground_Station_{sat}_NormVal_{year_list[0]}-{year_list[-1]}_Monthly.csv'
nv_df_m.to_csv(csv_path)
print(f"Saved normalization CSV to: {csv_path}")

nv_gdf_m = gpd.GeoDataFrame(nv_df_m, 
                            geometry=gpd.points_from_xy(nv_df_m['x'], nv_df_m['y']), 
                            crs='EPSG:4326').to_crs(CONFIG['target_crs'])

print("\nApplying normalization to satellite data...")
for pollutant in s5p_pollutants:
    print(f"  Normalizing {pollutant}...")
    apply_norm(nv_gdf_m, converted_zarr_path, sat, pollutant, 'yy_mm')
    gc.collect()

print(f"\n✓ {sat} normalization complete!")

NORMALIZING S5P DATA
Year range: 2024-2024
Pollutants: ['SO2', 'NO2', 'CO', 'O3']

--- Processing SO2 ---
Loading ground data...
Ground stations loaded: 240
Loading satellite data...
Satellite data loaded, shape: (12, 279, 503)
Sampling satellite data at station locations...
Samples matched: 240
Calculating normalization values...
Norm value range: -38.3675 to 23.3591

--- Processing NO2 ---
Loading ground data...
Ground stations loaded: 240
Loading satellite data...
Satellite data loaded, shape: (12, 279, 503)
Sampling satellite data at station locations...
Samples matched: 240
Calculating normalization values...
Norm value range: 1.1198 to 21.0946

--- Processing CO ---
Loading ground data...
Ground stations loaded: 84
Loading satellite data...
Satellite data loaded, shape: (12, 279, 503)
Sampling satellite data at station locations...
Samples matched: 84
Calculating normalization values...
Norm value range: 0.2117 to 5.0810

--- Processing O3 ---
Loading ground data...
Ground statio

## Normalize MOD Data

In [10]:
sat = 'MOD'
mod_pollutants = ['SO2', 'NO2', 'CO', 'O3', 'PM2P5', 'PM10']

print("\n" + "=" * 70)
print(f"NORMALIZING {sat} DATA")
print("=" * 70)
print(f"Pollutants: {mod_pollutants}")
print("=" * 70)

nv_df_list_m = []

for pollutant in mod_pollutants:
    print(f"\n--- Processing {pollutant} ---")
    gdf = load_ground_data(ground_station_csv, pollutant)
    da = load_sat_zarr(converted_zarr_path, sat, pollutant)
    gdf = sample_sat_data(da, gdf, pollutant)
    gdf[f"{pollutant}_Norm_Val"] = (gdf[pollutant] / gdf[f"{pollutant}_Sat"])
    nv_df_list_m.append(gdf)

nv_df_m = pd.concat(nv_df_list_m)[[f'{pollutant}_Norm_Val' for pollutant in mod_pollutants] + ['StationName', 'datetime', 'x', 'y']]
nv_df_m['yy_mm'] = pd.to_datetime(nv_df_m['datetime']).dt.strftime('%Y-%m')
nv_df_m = nv_df_m.drop(['datetime'], axis=1)
nv_df_m = nv_df_m.groupby(['yy_mm', 'StationName']).mean()

csv_path = converted_zarr_path / f'Ground_Station_{sat}_NormVal_{year_list[0]}-{year_list[-1]}_Monthly.csv'
nv_df_m.to_csv(csv_path)
print(f"\nSaved normalization CSV to: {csv_path}")

nv_gdf_m = gpd.GeoDataFrame(nv_df_m, 
                            geometry=gpd.points_from_xy(nv_df_m['x'], nv_df_m['y']), 
                            crs='EPSG:4326').to_crs(CONFIG['target_crs'])

print("\nApplying normalization...")
for pollutant in mod_pollutants:
    print(f"  Normalizing {pollutant}...")
    apply_norm(nv_gdf_m, converted_zarr_path, sat, pollutant, 'yy_mm')
    gc.collect()

print(f"\n✓ {sat} normalization complete!")


NORMALIZING MOD DATA
Pollutants: ['SO2', 'NO2', 'CO', 'O3', 'PM2P5', 'PM10']

--- Processing SO2 ---

--- Processing NO2 ---

--- Processing CO ---

--- Processing O3 ---

--- Processing PM2P5 ---

--- Processing PM10 ---

Saved normalization CSV to: C:\projects\aqi_scad\02_Input\Ground_Station_MOD_NormVal_2024-2024_Monthly.csv

Applying normalization...
  Normalizing SO2...
  Normalizing NO2...
  Normalizing CO...
  Normalizing O3...
  Normalizing PM2P5...
  Normalizing PM10...

✓ MOD normalization complete!


## Normalize CAMS Data

In [11]:
sat = 'CAMS'
cams_pollutants = ['SO2', 'NO2', 'CO', 'O3', 'PM2P5', 'PM10']

print("\n" + "=" * 70)
print(f"NORMALIZING {sat} DATA")
print("=" * 70)
print(f"Pollutants: {cams_pollutants}")
print("=" * 70)

nv_df_list_m = []

for pollutant in cams_pollutants:
    print(f"\n--- Processing {pollutant} ---")
    gdf = load_ground_data(ground_station_csv, pollutant)
    da = load_sat_zarr(converted_zarr_path, sat, pollutant)
    gdf = sample_sat_data(da, gdf, pollutant)
    gdf[f"{pollutant}_Norm_Val"] = (gdf[pollutant] / gdf[f"{pollutant}_Sat"])
    nv_df_list_m.append(gdf)

nv_df_m = pd.concat(nv_df_list_m)[[f'{pollutant}_Norm_Val' for pollutant in cams_pollutants] + ['StationName', 'datetime', 'x', 'y']]
nv_df_m['yy_mm'] = pd.to_datetime(nv_df_m['datetime']).dt.strftime('%Y-%m')
nv_df_m = nv_df_m.drop(['datetime'], axis=1)
nv_df_m = nv_df_m.groupby(['yy_mm', 'StationName']).mean()

csv_path = converted_zarr_path / f'Ground_Station_{sat}_NormVal_{year_list[0]}-{year_list[-1]}_Monthly.csv'
nv_df_m.to_csv(csv_path)
print(f"\nSaved normalization CSV to: {csv_path}")

nv_gdf_m = gpd.GeoDataFrame(nv_df_m, 
                            geometry=gpd.points_from_xy(nv_df_m['x'], nv_df_m['y']), 
                            crs='EPSG:4326').to_crs(CONFIG['target_crs'])

print("\nApplying normalization...")
for pollutant in cams_pollutants:
    print(f"  Normalizing {pollutant}...")
    apply_norm(nv_gdf_m, converted_zarr_path, sat, pollutant, 'yy_mm')
    gc.collect()

print(f"\n✓ {sat} normalization complete!")


NORMALIZING CAMS DATA
Pollutants: ['SO2', 'NO2', 'CO', 'O3', 'PM2P5', 'PM10']

--- Processing SO2 ---

--- Processing NO2 ---

--- Processing CO ---

--- Processing O3 ---

--- Processing PM2P5 ---

--- Processing PM10 ---

Saved normalization CSV to: C:\projects\aqi_scad\02_Input\Ground_Station_CAMS_NormVal_2024-2024_Monthly.csv

Applying normalization...
  Normalizing SO2...
  Normalizing NO2...
  Normalizing CO...
  Normalizing O3...
  Normalizing PM2P5...
  Normalizing PM10...

✓ CAMS normalization complete!


## Summary

In [12]:
print("\n" + "=" * 70)
print("NORMALIZATION SUMMARY")
print("=" * 70)
print(f"\nDate Range: {CONFIG['start_date']} to {CONFIG['end_date']}")
print(f"Satellites Normalized: S5P, MOD, CAMS")
print(f"\nOutput Location: {converted_zarr_path}")
print(f"Backup Location: {converted_zarr_path / 'Backup'}")
print("\n✓ All normalization complete!")
print("Ready for pollutant processing step!")
print("=" * 70)


NORMALIZATION SUMMARY

Date Range: 2024-01-01 to 2024-12-31
Satellites Normalized: S5P, MOD, CAMS

Output Location: C:\projects\aqi_scad\02_Input
Backup Location: C:\projects\aqi_scad\02_Input\Backup

✓ All normalization complete!
Ready for pollutant processing step!
